# SentinelAI — المرحلة ٢: تجهيز البيانات للتدريب (Preprocessing)

هذا الـ Notebook **لا يدرّب أي نموذج**. هدفه فقط تجهيز البيانات بحيث تصبح جاهزة
للتدريب في المرحلة القادمة. الخطوات:

1. تحميل البيانات المنظّفة وتطبيق أسماء الفئات المعروفة من توثيق `01_eda_cleaning.ipynb`.
2. فصل الخصائص (X) عن عمود الهدف (y = `Label`).
3. تقسيم البيانات (Train/Test) بطريقة Stratified للحفاظ على نسب الفئات النادرة.
4. توحيد مقياس الخصائص العددية (StandardScaler) — **بعد** التقسيم وليس قبله.
5. حساب أوزان الفئات (Class Weights) لمعالجة عدم توازن البيانات.
6. حفظ كل شيء (بيانات التدريب/الاختبار + أدوات التحويل) لاستخدامها في مرحلة التدريب.

In [1]:
import os
import glob
import json
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.utils.class_weight import compute_class_weight
import joblib

DATA_DIR = os.path.join("..", "data")
PROCESSED_DIR = os.path.join(DATA_DIR, "processed")
os.makedirs(PROCESSED_DIR, exist_ok=True)

RANDOM_STATE = 42

**ماذا فعلنا ولماذا؟**
استوردنا أدوات `scikit-learn` التي نحتاجها لهذه المرحلة: `train_test_split` لتقسيم
البيانات، `StandardScaler` لتوحيد مقياس الخصائص، و `compute_class_weight` لحساب أوزان
الفئات. استخدمنا `joblib` (يأتي مع scikit-learn، بدون تثبيت مكتبة إضافية) لحفظ الأدوات
المدرَّبة مثل الـ Scaler. حدّدنا `RANDOM_STATE = 42` ثابتاً حتى تكون النتائج قابلة
للتكرار في كل مرة نشغّل فيها الكود.

In [2]:
# نفس خطوات التنظيف من 01_eda_cleaning.ipynb، مكرّرة هنا حتى يكون هذا الملف
# مستقلاً وقابلاً للتشغيل وحده دون الاعتماد على تشغيل الملف الأول أولاً.
csv_files = sorted(glob.glob(os.path.join(DATA_DIR, "*.csv")))
if not csv_files:
    raise FileNotFoundError("لم يتم العثور على أي ملف CSV داخل مجلد data/.")

df = pd.concat([pd.read_csv(f, low_memory=False) for f in csv_files], ignore_index=True)
df.columns = df.columns.str.strip()

numeric_cols = df.select_dtypes(include=[np.number]).columns
df[numeric_cols] = df[numeric_cols].replace([np.inf, -np.inf], np.nan)
df = df.dropna()
df = df.drop_duplicates()

print(f"شكل البيانات بعد التنظيف: {df.shape[0]:,} صف × {df.shape[1]} عمود")

شكل البيانات بعد التنظيف: 1,135,409 صف × 79 عمود


**ماذا فعلنا ولماذا؟**
أعدنا نفس خطوات التنظيف التي شرحناها بالتفصيل في `01_eda_cleaning.ipynb` (تنظيف أسماء
الأعمدة، إزالة القيم اللانهائية/الناقصة، حذف التكرارات) للحصول على نفس البيانات النظيفة
(1,135,409 صف). النتيجة يجب أن تطابق ما رأيناه في الملف الأول.

In [3]:
# كما وثّقنا في 01_eda_cleaning.ipynb: لا يوجد عمود نصي بأسماء الهجمات في هذا الملف،
# والمعلومة الوحيدة شبه المؤكدة هي أن الرمز 1 (٧٧.٥١٪ من البيانات) هو الفئة الطبيعية.
# بقية الرموز (2 إلى 11) نتركها بأسماء عامة "Attack_<code>" لأننا لا نعرف أسماءها الحقيقية
# بدون ملف mapping خارجي أو سكربت التحضير الأصلي.
label_names = {1: "Benign"}
for code in sorted(df["Label"].unique()):
    label_names.setdefault(code, f"Attack_{code}")

df["Label_Name"] = df["Label"].map(label_names)

print("الأسماء المطبَّقة على الفئات:")
for code, name in sorted(label_names.items()):
    count = (df["Label"] == code).sum()
    print(f"  {code:>2} -> {name:<12} ({count:,} صف)")

الأسماء المطبَّقة على الفئات:
   1 -> Benign       (880,060 صف)
   2 -> Attack_2     (35,127 صف)
   3 -> Attack_3     (33,817 صف)
   4 -> Attack_4     (124,280 صف)
   5 -> Attack_5     (52,051 صف)
   6 -> Attack_6     (7,598 صف)
   7 -> Attack_7     (2,028 صف)
   8 -> Attack_8     (301 صف)
   9 -> Attack_9     (91 صف)
  10 -> Attack_10    (46 صف)
  11 -> Attack_11    (10 صف)


**ماذا فعلنا ولماذا؟**
طبّقنا فقط ما هو **موثَّق وموثوق فعلاً**: الرمز `1` = `Benign`. أما الرموز من `2` إلى `11`
فلا يوجد لدينا مصدر يؤكّد اسم الهجوم الحقيقي لكل منها (لا عمود نصي في الملف، ولا ملف
mapping خارجي)، لذلك سمّيناها بأسماء عامة `Attack_2`, `Attack_3`, ... `Attack_11` بدل أن
نخترع أسماء هجمات قد تكون غير صحيحة. عمود `Label_Name` هذا **للعرض والتوضيح فقط** — التدريب
سيستخدم عمود `Label` الرقمي الأصلي.

In [4]:
non_feature_cols = ["Unnamed: 0", "Label", "Label_Name"]
feature_cols = [c for c in df.columns if c not in non_feature_cols]

X = df[feature_cols].copy()
y = df["Label"].copy()

print(f"عدد الخصائص (X): {X.shape[1]}")
print(f"عدد صفوف X: {X.shape[0]:,} — عدد صفوف y: {y.shape[0]:,}")

عدد الخصائص (X): 77
عدد صفوف X: 1,135,409 — عدد صفوف y: 1,135,409


**ماذا فعلنا ولماذا؟**
فصلنا الخصائص (X) عن الهدف (y). استبعدنا من X عمود `Unnamed: 0` لأنه مجرد رقم صف أصلي
من الملف الخام وليس خاصية حقيقية للتدفّق (لو تركناه قد يعلَّم النموذج نمطاً وهمياً لا معنى
له)، واستبعدنا `Label` و `Label_Name` لأنهما الهدف نفسه، ووجودهما داخل X يُعتبر تسريب
بيانات (Data Leakage) يجعل النتائج تبدو ممتازة كذباً.

In [5]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,
    random_state=RANDOM_STATE,
)

print(f"حجم بيانات التدريب: {X_train.shape[0]:,} صف")
print(f"حجم بيانات الاختبار: {X_test.shape[0]:,} صف")

compare = pd.DataFrame({
    "النسبة % - الكل": (y.value_counts(normalize=True) * 100).round(3),
    "النسبة % - Train": (y_train.value_counts(normalize=True) * 100).round(3),
    "النسبة % - Test": (y_test.value_counts(normalize=True) * 100).round(3),
}).sort_index()
print("\nمقارنة نسب الفئات قبل/بعد التقسيم:")
print(compare)

حجم بيانات التدريب: 908,327 صف
حجم بيانات الاختبار: 227,082 صف

مقارنة نسب الفئات قبل/بعد التقسيم:
       النسبة % - الكل  النسبة % - Train  النسبة % - Test
Label                                                    
1               77.510            77.510           77.510
2                3.094             3.094            3.094
3                2.978             2.978            2.978
4               10.946            10.946           10.946
5                4.584             4.584            4.584
6                0.669             0.669            0.669
7                0.179             0.179            0.179
8                0.027             0.027            0.026
9                0.008             0.008            0.008
10               0.004             0.004            0.004
11               0.001             0.001            0.001


**ماذا فعلنا ولماذا؟ (وأين اختلفنا عن ترتيب الخطوات المطلوب حرفياً)**
قسّمنا البيانات إلى تدريب (80%) واختبار (20%) باستخدام `stratify=y`، وهذا يعني أن **نسبة
كل فئة تبقى نفسها تقريباً** في التدريب والاختبار — حتى الفئات النادرة جداً مثل الرمز `11`
(10 صفوف فقط في كل البيانات) ستُوزَّع بنفس النسبة تقريباً، بدل أن تختفي بالصدفة من أحد
الطرفين.

لاحظ أننا قمنا **بالتقسيم قبل التوحيد القياسي (Scaling)**، رغم أن الترتيب المطلوب ذكر
التوحيد القياسي أولاً ثم معالجة عدم التوازن. غيّرنا هذا الترتيب عمداً لسبب تقني مهم: لو
دربنا الـ `StandardScaler` على كل البيانات (تدريب + اختبار معاً)، سيكون قد "رأى" إحصائيات
بيانات الاختبار قبل الأوان، وهذا يُسمى **تسريب بيانات (Data Leakage)** ويجعل تقييم النموذج
لاحقاً متفائلاً بشكل غير واقعي. لذلك الترتيب الصحيح تقنياً هو: تقسيم أولاً، ثم توحيد قياسي
مبني فقط على بيانات التدريب.

In [6]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("قبل التوحيد القياسي (عيّنة من التدريب):")
print(X_train.iloc[:, :3].describe().loc[["mean", "std"]])

print("\nبعد التوحيد القياسي (نفس الأعمدة، متوقّع تقريباً mean=0, std=1):")
print(pd.DataFrame(X_train_scaled[:, :3], columns=X_train.columns[:3]).describe().loc[["mean", "std"]])

قبل التوحيد القياسي (عيّنة من التدريب):
          Dst Port  Protocol  Flow Duration
mean   9590.750127  8.312495   1.093227e+07
std   19358.289926  4.618650   8.917787e+08

بعد التوحيد القياسي (نفس الأعمدة، متوقّع تقريباً mean=0, std=1):
          Dst Port      Protocol  Flow Duration
mean -4.399398e-17  2.062022e-16   6.883838e-19
std   1.000001e+00  1.000001e+00   1.000001e+00


**ماذا فعلنا ولماذا؟**
درّبنا (`fit`) الـ `StandardScaler` على بيانات التدريب فقط، ثم استخدمناه لتحويل
(`transform`) كلاً من التدريب والاختبار بنفس القيم (المتوسط والانحراف المعياري) المحسوبة
من التدريب. هذا يجعل كل خاصية بمتوسط قريب من صفر وانحراف معياري قريب من واحد، وهو مهم لأن
خصائص هذا الـ Dataset متفاوتة جداً في مقياسها (مثلاً `Flow Duration` بالميكروثانية مقابل
أعلام TCP التي قيمتها 0 أو 1).

In [7]:
classes = np.unique(y_train)
weights = compute_class_weight(class_weight="balanced", classes=classes, y=y_train)
class_weights = {int(c): float(w) for c, w in zip(classes, weights)}

weights_table = pd.DataFrame({
    "Label": list(class_weights.keys()),
    "الاسم": [label_names[c] for c in class_weights.keys()],
    "عدد صفوف Train": [int((y_train == c).sum()) for c in class_weights.keys()],
    "الوزن": list(class_weights.values()),
}).sort_values("Label")

print(weights_table.to_string(index=False))

 Label     الاسم  عدد صفوف Train        الوزن
     1    Benign          704048     0.117286
     2  Attack_2           28101     2.938514
     3  Attack_3           27054     3.052236
     4  Attack_4           99424     0.830536
     5  Attack_5           41641     1.983026
     6  Attack_6            6078    13.585913
     7  Attack_7            1622    50.909483
     8  Attack_8             241   342.635609
     9  Attack_9              73  1131.166874
    10 Attack_10              37  2231.761671
    11 Attack_11               8 10321.897727


**ماذا فعلنا ولماذا؟ (طريقة معالجة عدم توازن الفئات المختارة)**
استخدمنا `stratify` في التقسيم (خطوة سابقة) **بالإضافة إلى** حساب أوزان الفئات
(Class Weights) بدل أخذ عيّنة متوازنة (Undersampling) أو تكرار/توليد بيانات صناعية
(Oversampling / SMOTE). الأسباب:

1. **الحفاظ على كل البيانات:** لدينا فئات نادرة جداً (الرمز `11` فيه 10 صفوف فقط في كل
   الـ Dataset، والرمز `10` فيه 46 صفاً). لو أخذنا عيّنة متوازنة بحجم أصغر فئة، سنفقد
   تقريباً كل بيانات فئة `Benign` (880 ألف صف) وهذا هدر كبير.
2. **بدون بيانات صناعية:** تقنيات مثل SMOTE تحتاج مكتبة إضافية (`imbalanced-learn`) غير
   موجودة في متطلبات المشروع، وتوليد بيانات صناعية لفئة فيها 10 صفوف فقط قد ينتج بيانات
   غير واقعية.
3. **أوزان الفئات** تخبر النموذج لاحقاً (في مرحلة التدريب) أن يعطي "اهتماماً" أكبر للفئات
   النادرة عند حساب الخطأ، دون حذف أو تكرار أي صف فعلي. الفئة الأكبر (`Benign`) ستحصل على
   وزن صغير، والفئات النادرة جداً ستحصل على وزن كبير جداً.

هذه الأوزان محفوظة الآن لتُستخدم في مرحلة التدريب القادمة (كمعامل `class_weight` في
خوارزميات مثل Random Forest / Logistic Regression، أو داخل دالة الخسارة في CNN).

In [8]:
np.save(os.path.join(PROCESSED_DIR, "X_train.npy"), X_train_scaled)
np.save(os.path.join(PROCESSED_DIR, "X_test.npy"), X_test_scaled)
np.save(os.path.join(PROCESSED_DIR, "y_train.npy"), y_train.to_numpy())
np.save(os.path.join(PROCESSED_DIR, "y_test.npy"), y_test.to_numpy())

with open(os.path.join(PROCESSED_DIR, "feature_names.json"), "w", encoding="utf-8") as f:
    json.dump(feature_cols, f, ensure_ascii=False, indent=2)

with open(os.path.join(PROCESSED_DIR, "label_mapping.json"), "w", encoding="utf-8") as f:
    json.dump({str(k): v for k, v in label_names.items()}, f, ensure_ascii=False, indent=2)

with open(os.path.join(PROCESSED_DIR, "class_weights.json"), "w", encoding="utf-8") as f:
    json.dump({str(k): v for k, v in class_weights.items()}, f, ensure_ascii=False, indent=2)

joblib.dump(scaler, os.path.join(PROCESSED_DIR, "scaler.joblib"))

print("تم حفظ الملفات التالية في data/processed/:")
for fname in sorted(os.listdir(PROCESSED_DIR)):
    fpath = os.path.join(PROCESSED_DIR, fname)
    size_mb = os.path.getsize(fpath) / (1024 * 1024)
    print(f"  - {fname} ({size_mb:.2f} MB)")

تم حفظ الملفات التالية في data/processed/:
  - X_test.npy (133.40 MB)
  - X_train.npy (533.61 MB)
  - class_weights.json (0.00 MB)
  - feature_names.json (0.00 MB)
  - label_mapping.json (0.00 MB)
  - scaler.joblib (0.00 MB)
  - y_test.npy (1.73 MB)
  - y_train.npy (6.93 MB)


**ماذا فعلنا ولماذا؟**
حفظنا كل ما تحتاجه مرحلة التدريب القادمة داخل `data/processed/` حتى لا تحتاج إعادة تنفيذ
كل هذا الملف من جديد:

- `X_train.npy` / `X_test.npy` / `y_train.npy` / `y_test.npy`: البيانات جاهزة للتدريب
  والاختبار (مُقاسة بالفعل عبر StandardScaler).
- `feature_names.json`: ترتيب أسماء الأعمدة في X (مفيد لتفسير أهمية الخصائص لاحقاً).
- `label_mapping.json`: تعريف كل رمز فئة (1 = Benign، والباقي Attack_N كما وثّقنا).
- `class_weights.json`: الأوزان الجاهزة لتمريرها للنماذج في مرحلة التدريب.
- `scaler.joblib`: أداة التوحيد القياسي المدرَّبة، ليتم استخدامها لاحقاً على أي بيانات
  جديدة (مثلاً عند تحليل تدفّق جديد عبر الـ API) بنفس المقياس تماماً.

## الخلاصة

جهّزنا البيانات بالكامل للتدريب:
- طبّقنا أسماء الفئات المعروفة (Benign فقط مؤكّد، والباقي بأسماء عامة موثَّقة كـ "غير معروف").
- فصلنا X عن y وأزلنا الأعمدة غير المفيدة.
- قسّمنا البيانات بطريقة Stratified (80% تدريب / 20% اختبار).
- طبّقنا StandardScaler بعد التقسيم (لتفادي تسريب البيانات).
- حسبنا أوزان الفئات لمعالجة عدم التوازن دون فقدان أو اختلاق بيانات.
- حفظنا كل الملفات اللازمة في `data/processed/`.

**لم يتم تدريب أي نموذج بعد.** الخطوة التالية هي تدريب الخوارزميات السبع باستخدام هذه
الملفات المحفوظة.